# 21 · POPE-vi — QLoRA có làm tăng hallucination vật thể không? (H4, W12)

**Vấn đề:** EM/ANLS chỉ cho biết đáp án đúng hay sai. Chúng không cho biết model có **bịa ra vật thể không có trong ảnh** hay không. Hội đồng sẽ hỏi: "Fine-tune QLoRA có làm model ảo giác nhiều hơn không?" Đó là giả thuyết **H4** (H8 cũ ở W02).

**Cách đo (POPE, Li et al., EMNLP 2023, làm bằng tiếng Việt):** hỏi `Trong ảnh có {vật} không? Trả lời "có" hoặc "không".`
- Câu **"có"**: 3 vật có thật trong ảnh, lấy từ annotation COCO 2014 (ảnh ViVQA chính là ảnh COCO).
- Câu **"không"**: 3 vật không có trong ảnh, chọn theo 3 cách: **random** (ngẫu nhiên), **popular** (vật phổ biến nhất), **adversarial** (vật hay đi kèm với vật có trong ảnh, khó nhất).
- 500 ảnh ViVQA test **chưa từng xuất hiện trong train**, seed 42.

**Luật ghi trước** (`docs/W12/README.md`, mục H4, commit TRƯỚC khi chạy): ΔF1 = F1(adapter) − F1(zero-shot cùng prompt), CI95 bootstrap theo ảnh.
- **KHÔNG TĂNG HALLUCINATION**: cận dưới CI95 > −2 ở cả 3 cách chọn.
- **TĂNG HALLUCINATION**: cận trên CI95 < −2 ở ít nhất 1 cách chọn.
- Còn lại: **CHƯA KẾT LUẬN**.

| So sánh | Mốc (cùng prompt) |
|---|---|
| C3 (adapter ViVQA chính thức) | B1-plain (zero-shot, prompt thường) |
| B3-full (adapter ViTextVQA chính thức) | B1-ocr (zero-shot, prompt có dòng OCR) |
| SH (adapter prototype, báo riêng) | B1-ocr |

| Phần | Runtime | Việc | Thời gian |
|---|---|---|---|
| A | **CPU** (rẻ) | tải annotation COCO, sinh file câu hỏi | ≈ 10 phút |
| — | — | ⏸ **ĐIỂM DỪNG A**: Cursor chép file câu hỏi vào repo + ghi sha256 vào README + commit | — |
| B | **L4** | 5 cấu hình × ≈ 5.400 câu (≈ 0,57 s/câu) | ≈ 4,5 giờ (≈ 7 CU) |
| B | L4 | chấm + phán quyết, tự trả máy | ≈ 1 phút |

## 1. Thiết lập môi trường (chạy được cả CPU lẫn L4)

Mount Drive, lấy code mới nhất từ GitHub, khai báo đường dẫn và tên run. Cell này **không** đòi GPU. Phần A chạy trên CPU cho đỡ tốn CU, còn Phần B tự kiểm L4 ở mục 4.

*Kỳ vọng: commit mới nhất (có notebook 21), dòng `Runtime: CPU` (Phần A) hoặc `Runtime: NVIDIA L4` (Phần B), `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, subprocess
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('src/eval/pope_vi.py'), 'Code cũ -> git pull chưa lấy được commit W12 (pope_vi.py)'

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
VIVQA_DIR = f'{DRIVE_ROOT}/data/vivqa'
Q_NAME = 'pope_vi_questions_v1.json'
Q_DRIVE = f'{EXP_ROOT}/W12_pope_vi_questions/{Q_NAME}'      # bản sinh trên Colab
Q_REPO = f'experiments/W12_pope_vi_questions/{Q_NAME}'      # bản đã commit (luật ghi trước)
OUT = f'{EXP_ROOT}/W12_pope_vi_L4_seed42'
CHUNK = 500
RUNS = {   # nhãn -> tên run (thư mục trong experiments/)
    'B1_plain': 'W12_POPE_b1_zeroshot_qwen25vl_popevi_v1_bf16_L4_seed42',
    'C3':       'W12_POPE_c3_qlora_r16_vivqa_ep2_popevi_v1_bf16_L4_seed42',
    'B1_ocr':   'W12_POPE_b1_ocr_vietocr_v1_qwen25vl_popevi_v1_bf16_L4_seed42',
    'B3full':   'W12_POPE_b3full_qlora_ocr_vietocr_v1_r16_popevi_v1_bf16_L4_seed42',
    'SH':       'W12_POPE_sh_qlora_ocr_vietocr_v1_r16_shared_popevi_v1_bf16_L4_seed42',
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

def run_cmd(cmd):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}) -> xem log ở trên, gửi Claude')

GPU = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip() or 'CPU'
print('Runtime:', GPU)
print('✅ Thiết lập xong')

## 2. Phần A — Tải annotation COCO 2014 (chỉ cần khi chưa có file câu hỏi)

Tải `annotations_trainval2014.zip` (≈ 250 MB) từ máy chủ COCO về **đĩa tạm của Colab** (`/content`), rồi chỉ giải nén 2 file `instances_train2014.json` và `instances_val2014.json`. File **không** chép lên Drive, để Drive gọn. sha256 của 2 file được ghi vào file câu hỏi để truy vết.

Nếu file câu hỏi đã có trên Drive (chạy lại, hoặc đang ở Phần B) thì cell tự bỏ qua.

*Kỳ vọng: `instances_train2014.json … MB` và `instances_val2014.json … MB`, hoặc `⏭ đã có file câu hỏi -> bỏ qua`. Tải mất khoảng 1–3 phút.*

In [ ]:
COCO_DIR = '/content/coco_ann'
if os.path.exists(Q_DRIVE):
    print('⏭ đã có file câu hỏi -> bỏ qua tải COCO')
else:
    os.makedirs(COCO_DIR, exist_ok=True)
    if not os.path.exists('/content/annotations_trainval2014.zip'):
        !wget -q --show-progress -O /content/annotations_trainval2014.zip http://images.cocodataset.org/annotations/annotations_trainval2014.zip
    !unzip -j -o -q /content/annotations_trainval2014.zip annotations/instances_train2014.json annotations/instances_val2014.json -d {COCO_DIR}
    for f in ('instances_train2014.json', 'instances_val2014.json'):
        p = f'{COCO_DIR}/{f}'
        print(f'{f}  {os.path.getsize(p) / 1e6:.0f} MB')

## 3. Phần A — Sinh file câu hỏi POPE-vi

Trước hết chạy selftest của `pope_vi.py`: đọc câu trả lời, loại ảnh train, loại lớp dễ nhầm, phán quyết. Sau đó sinh file câu hỏi **một lần duy nhất**. Nếu file đã có thì **không** sinh lại, vì đây là bộ câu hỏi đã chốt.

Cặp (ảnh, vật) trùng giữa các cách chọn chỉ hỏi **một lần**. Model chạy greedy nên cùng prompt luôn ra cùng đáp án, và cách này tiết kiệm GPU.

Sau đó so với bản đã commit trong repo:
- **Chưa có bản commit** → ⏸ **ĐIỂM DỪNG A**: gửi Claude output cell này. Claude đưa prompt để Cursor chép file từ Drive vào repo, ghi sha256 vào README rồi commit + push. Đó là bước khoá luật trước khi chạy model.
- **Đã có và trùng sha256** → `✅`, sang Phần B.

*Kỳ vọng (lần đầu): `pope_vi selftest OK`, dòng `✔ … câu duy nhất | 500 ảnh …`, 8 câu mẫu, `sha256 …`, rồi `⏸ ĐIỂM DỪNG A`.*

In [ ]:
run_cmd('python -m src.eval.pope_vi --selftest')
if not os.path.exists(Q_DRIVE):
    run_cmd(f'python -m src.eval.pope_vi --build --coco_ann_dir {COCO_DIR} --vivqa_dir {VIVQA_DIR} --out {Q_DRIVE}')
Q_SHA = sha256_file(Q_DRIVE)
st = json.load(open(Q_DRIVE, encoding='utf-8'))['stats']
print(f"File câu hỏi: {st['unique_questions']} câu | {st['chosen_images']} ảnh | mỗi cách chọn {st['items_per_setting']} | sha256 {Q_SHA}")
if not os.path.exists(Q_REPO):
    raise SystemExit('⏸ ĐIỂM DỪNG A: chưa có bản commit trong repo -> gửi Claude output cell này, chạy prompt Cursor, '
                     'rồi đổi runtime sang L4 và Run all lại.')
assert sha256_file(Q_REPO) == Q_SHA, '❌ File câu hỏi trên Drive KHÁC bản đã commit -> dừng, báo Claude'
print('✅ File câu hỏi trùng bản đã commit (luật ghi trước) -> sang Phần B')

## 4. Phần B — Kiểm tra trước khi tốn ≈ 4,5 giờ L4

Phần này **bắt buộc L4**, cùng GPU với mọi run gốc. Nếu đang ở CPU thì vào **Runtime → Change runtime type → L4 → Save**, rồi **Runtime → Run all**. Các cell Phần A sẽ tự bỏ qua.

Các bước kiểm:
- **(a)** Cài thư viện, cùng pin `transformers==5.17.0` như các notebook train/eval trước.
- **(b)** Cấu hình OCR đọc từ `ocr_selection.json` (W08), **không gõ tay**. OCR cache ViVQA phải trùng sha256 biên bản W11 và phủ đủ ảnh POPE.
- **(c)** Ba run gốc C3, B3-full, SH có trên Drive. Adapter trong thư mục ghi ở run gốc phải **trùng dấu vân tay trọng số**.
- **(d)** Selftest `run_baseline` + `pope_vi`.

*Kỳ vọng: `GPU: NVIDIA L4`, `paddledet_vietocr | v1 | 600 | 0.0`, `OCR cache ViVQA = biên bản W11 ✅`, `ảnh POPE có OCR: 500/500`, 3 dòng `adapter … = run gốc ✅`, `✅ selftest OK (2 script)`.*

In [ ]:
if 'L4' not in GPU:
    raise SystemExit('❌ Phần B phải chạy trên L4 (cùng GPU với C3 / B3-full / SH). '
                     'Runtime -> Change runtime type -> L4, rồi Run all.')
print('GPU:', GPU)
!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

from src.data.ocr import default_cache_path, load_ocr_cache
from src.eval.run_baseline import adapter_fingerprint
from src.eval.pope_vi import REF_EVAL

# (b) OCR: cấu hình đóng băng W08 + cache ViVQA đúng biên bản W11 + phủ đủ ảnh POPE
sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'
C_VV = default_cache_path(DRIVE_ROOT, 'vivqa', BACKEND)
MAN_VV = f'{EXP_ROOT}/W11_ocr_cache_vivqa/ocr_cache_manifest.json'
assert sha256_file(C_VV) == json.load(open(MAN_VV, encoding='utf-8'))['sha256'], 'OCR cache ViVQA KHÁC biên bản W11 -> dừng, báo Claude'
print('OCR cache ViVQA = biên bản W11 ✅')
qs = json.load(open(Q_DRIVE, encoding='utf-8'))['data']
need, have = {q['image'] for q in qs}, set(load_ocr_cache(C_VV)['images'])
print(f'ảnh POPE có OCR: {len(need & have)}/{len(need)}')
assert need <= have, 'OCR cache thiếu ảnh POPE -> báo Claude'

# (c) adapter = đúng adapter của run gốc (so dấu vân tay trọng số, không so đường dẫn)
AD = {}
for k, run in REF_EVAL.items():
    ref = json.load(open(f'{EXP_ROOT}/{run}/metrics.json', encoding='utf-8'))['meta']['adapter']
    fp = adapter_fingerprint(ref['path'])
    assert fp['weights_sha256'] == ref['weights_sha256'], f'{k}: adapter trên Drive KHÁC run gốc -> dừng, báo Claude'
    AD[k] = ref['path']
    print(f"adapter {k:6s} = run gốc ✅  ({fp['weights_mb']} MB, {fp['weights_sha256'][:12]}…)  {ref['path']}")

# (d) selftest
for mod, must in {'src.eval.run_baseline': 'SELFTEST OK', 'src.eval.pope_vi': 'pope_vi selftest OK'}.items():
    o = subprocess.run(f'python -m {mod} --selftest', shell=True, capture_output=True, text=True)
    assert must in o.stdout, (mod, o.stdout[-800:], o.stderr[-800:])
print('✅ selftest OK (2 script)')

## 5. Phần B — Chạy 5 cấu hình (≈ 50 phút mỗi cấu hình)

Mỗi cấu hình là một lần `run_baseline` trên file câu hỏi POPE. Thứ tự chạy: B1-plain → C3 → B1-ocr → B3-full → SH. Prompt thường chạy trước, prompt OCR chạy sau.

Mỗi lô 500 câu được ghi lên Drive ngay khi xong. **Bị ngắt thì Run all lại**: cấu hình đã xong tự bỏ qua, cấu hình đang dở chạy tiếp từ lô cuối. Giữ máy không ngủ và tab mở.

*Kỳ vọng: với mỗi cấu hình có dòng `== KẾT QUẢ: EM=…` (EM ở đây = tỉ lệ trả lời đúng có/không, chỉ để kiểm nhanh), sau đó `✔ <nhãn>: … câu | … s/câu`. Cấu hình dùng OCR có thêm dòng `OCR: …/data/vivqa/ocr/paddledet_vietocr.json`. Mỗi cấu hình xong là một **điểm dừng an toàn**.*

In [ ]:
PLAN = {   # nhãn -> (adapter, dùng OCR?)
    'B1_plain': (None, False), 'C3': (AD['C3'], False),
    'B1_ocr': (None, True), 'B3full': (AD['B3full'], True), 'SH': (AD['SH'], True),
}
for label, (adapter, use_ocr) in PLAN.items():
    run = RUNS[label]
    if os.path.exists(f'{EXP_ROOT}/{run}/metrics.json'):
        print(f'⏭ {label}: đã có -> bỏ qua')
    else:
        cmd = f'python -u -m src.eval.run_baseline --dataset vivqa --data_file {Q_DRIVE} --run_name {run} --chunk {CHUNK}'
        if adapter:
            cmd += f' --adapter {adapter}'
        if use_ocr:
            cmd += f' {OCR_ARGS}'
        run_cmd(cmd)
    m = json.load(open(f'{EXP_ROOT}/{run}/metrics.json', encoding='utf-8'))['meta']
    print(f"✔ {label}: {m['num_samples']} câu | {m['sec_per_sample']} s/câu | {m['inference_time_sec'] / 3600:.2f} h\n")

## 6. Kết luận theo luật ghi trước

`src.eval.pope_vi --score` làm ba việc:
1. **Kiểm hợp lệ** từng cấu hình: đủ câu, đúng sha256 file câu hỏi, đúng mode, adapter trùng run gốc, OCR cùng thiết lập với SH. Sai điều nào thì in ❌ và **không** kết luận.
2. **Tính metric** theo từng cách chọn: Accuracy, Precision, Recall, **F1**, yes-ratio, tỉ lệ không đọc được. Câu không đọc được tính **sai**. Cấu hình có > 5% câu không đọc được thì không dùng cho H4.
3. **Phán quyết** từng adapter và **H4 tổng**. Ngoài ra có phần chỉ mô tả: B1-ocr so với B1-plain.

*Kỳ vọng: bảng 15 dòng (5 cấu hình × 3 cách chọn), bảng 3 so sánh có phán quyết, dòng `➜ H4 THEO LUẬT: …`. File `pope_vi_summary.json` + `.md` nằm trong `experiments/W12_pope_vi_L4_seed42/`.*

In [ ]:
os.makedirs(OUT, exist_ok=True)
args = ' '.join(f'--run {k}={v}' for k, v in RUNS.items())
run_cmd(f'python -m src.eval.pope_vi --score --questions {Q_DRIVE} --exp_root {EXP_ROOT} {args} --out {OUT}')

## 7. Tự trả máy L4 khi chạy xong

Kết quả đã nằm trên Drive. Output phía trên vẫn hiện trong tab sau khi trả máy. Nếu notebook dừng sớm vì lỗi thì máy **không** tự trả: đọc lỗi xong, tự bấm **Runtime → Disconnect and delete runtime**.

*Kỳ vọng: `✅ Xong toàn bộ -> tự trả máy …`.*

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')

## 8. ⏸ DỪNG

Gửi Claude output **mục 6**. Nếu lỡ đóng tab mất output thì không sao: kết quả nằm trong `experiments/W12_POPE_*` và `W12_pope_vi_L4_seed42/` trên Drive, Cursor chép và kiểm lại được.